# Latihan 1: Tim Koordinator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-genai-migas/blob/main/colab/hari4/08-sistem-multi-agen-latihan-1.ipynb)

Ini versi **Google Colab**. Isi dan urutan contohnya sama dengan versi VS Code; yang berbeda hanya cell Persiapan di bawah, yang menyalin repository, memasang pustaka, dan mengisi API key di runtime Colab.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/dteti-workshop/workshop-genai-migas/blob/main/hari4/08-sistem-multi-agen/08-sistem-multi-agen.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-genai-migas.git"
REPO = "/content/workshop-genai-migas" if os.path.isdir("/content") else os.path.abspath("workshop-genai-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Supervisor mengoordinasi tiga spesialis:
- operasi: status sumur dan jadwal pulling
- pemeliharaan: work order terbuka
- engineering: water cut (HANYA peran engineer dan management, dipasang di kode)

Area dan peran pengguna datang dari konteks runtime, bukan dari model. Skrip juga melaporkan spesialis
mana yang dipakai tiap permintaan dan berapa token yang dihabiskan tim.

In [ ]:
# Langkah 1 - Menyiapkan model
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import ToolRuntime
from langchain_core.callbacks import UsageMetadataCallbackHandler
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from migas_shared.rag_tools import KonteksPengguna

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Data fiktif dan tool domain
Data dua area (Seruni dan Kenanga) beserta empat tool tingkat rendah untuk para spesialis.

In [ ]:
# ---------- Data fiktif untuk dua area ----------
SUMUR = {
    "Seruni": {
        "SR-020": "produksi 120 bopd, air 480 bwpd, ESP trip overload kemarin, stabil sejak 18.00",
        "SR-027": "produksi 95 bopd, air 380 bwpd, pembacaan H2S 12 ppm dalam investigasi HSE",
    },
    "Kenanga": {
        "KN-033": "tidak berproduksi, sucker rod patah, menunggu jadwal rig",
    },
}
JADWAL_PULLING = {
    "Seruni": "SR-020 ganti motor ESP Kamis 14.00",
    "Kenanga": "KN-033 pulling sucker rod Selasa 08.00",
}
WORK_ORDER = {
    "Seruni": "WO-26-12501 kebocoran seal ESP SR-020 (tinggi); WO-26-12502 lampu manifold berkedip (rendah)",
    "Kenanga": "WO-26-12611 sucker rod KN-033 patah, menunggu rig (tinggi)",
}


@tool
def cek_status_sumur(kode_sumur: str, area: str) -> str:
    """Cek status sebuah sumur di sebuah area.

    Args:
        kode_sumur: Kode sumur, contoh "SR-020"
        area: Nama area, contoh "Seruni"
    """
    if area not in SUMUR:
        return f"Area '{area}' tidak dikenal. Area yang ada: {', '.join(SUMUR)}"
    status = SUMUR[area].get(kode_sumur.upper())
    if status is None:
        # Pesan error yang membantu lebih baik daripada angka 0 yang menyesatkan: model bisa memperbaiki dirinya
        return f"Sumur '{kode_sumur}' tidak ada di area {area}. Sumur di area ini: {', '.join(SUMUR[area])}"
    return f"{kode_sumur.upper()} ({area}): {status}"


@tool
def ambil_jadwal_pulling(area: str) -> str:
    """Ambil jadwal pulling atau workover di sebuah area.

    Args:
        area: Nama area
    """
    return f"{area}: {JADWAL_PULLING.get(area, 'tidak ada jadwal')}"


@tool
def daftar_wo_terbuka(area: str) -> str:
    """Daftar work order terbuka di sebuah area.

    Args:
        area: Nama area
    """
    return f"{area}: {WORK_ORDER.get(area, 'tidak ada work order terbuka')}"


@tool
def hitung_water_cut(minyak_bopd: float, air_bwpd: float) -> str:
    """Hitung water cut dari produksi minyak dan air.

    Args:
        minyak_bopd: Produksi minyak (bopd)
        air_bwpd: Produksi air (bwpd)
    """
    total = minyak_bopd + air_bwpd
    if total == 0:
        return "Tidak dapat dihitung: total cairan 0."
    return f"Water cut {air_bwpd / total * 100:.1f}% (total cairan {total:.0f} bpd)"

## Langkah 3 - Agen spesialis
Tiap spesialis memegang tool-nya sendiri. `tanya_spesialis` memanggil agen dan tidak menjatuhkan seluruh tim kalau gagal.

In [ ]:
def tanya_spesialis(agen, teks: str) -> str:
    """Panggil agen spesialis. Jika gagal, kembalikan pesan, jangan menjatuhkan seluruh tim."""
    try:
        return agen.invoke({"messages": [HumanMessage(content=teks)]})["messages"][-1].content
    except Exception as error:  # misalnya model tidak tersedia, bug tool
        return f"Spesialis tidak tersedia ({type(error).__name__}). Minta pengguna mencoba lagi nanti."


agen_operasi = create_agent(
    model,
    tools=[cek_status_sumur, ambil_jadwal_pulling],
    system_prompt="Anda spesialis Operasi. Pakai tool dengan area yang diberikan. 1-2 kalimat.",
)
agen_pemeliharaan = create_agent(
    model,
    tools=[daftar_wo_terbuka],
    system_prompt="Anda spesialis Pemeliharaan. Pakai tool dengan area yang diberikan. 1-2 kalimat.",
)
agen_engineering = create_agent(
    model,
    tools=[hitung_water_cut],
    system_prompt="Anda spesialis Engineering. Hitung dengan tool, jangan menebak. 1-2 kalimat.",
)

## Langkah 4 - Tool wrapper dan koordinator
Area dan peran dibaca dari `runtime.context`. Akses Engineering dibatasi di kode.

In [ ]:
# TODO(B8.1): buat tool tanya_spesialis_operasi(permintaan, runtime: ToolRuntime[KonteksPengguna]) yang meneruskan permintaan ke agen_operasi dengan awalan area dari runtime.context.area
tanya_spesialis_operasi = None  # TODO: ganti dengan implementasi Anda


@tool
def tanya_spesialis_pemeliharaan(permintaan: str, runtime: ToolRuntime[KonteksPengguna]) -> str:
    """Tanya tentang work order terbuka dan peralatan rusak di area pengguna."""
    return tanya_spesialis(agen_pemeliharaan, f"Area {runtime.context.area}. {permintaan}")


# TODO(B8.2): buat tool tanya_spesialis_engineering yang menolak peran selain engineer dan management di kode (kembalikan pesan 'Akses ditolak'), lalu meneruskan permintaan ke agen_engineering
tanya_spesialis_engineering = None  # TODO: ganti dengan implementasi Anda


# TODO(B8.3): buat agen koordinator dengan ketiga tool wrapper, context_schema=KonteksPengguna, dan system prompt: tidak memegang data sendiri, tanya spesialis yang relevan, laporkan akses ditolak, jangan mengarang angka
tim = None  # TODO: ganti dengan implementasi Anda

## Langkah 5 - Fungsi untuk satu pertanyaan
`tanya_tim` melaporkan spesialis mana yang ditanya, jumlah token, dan jawabannya.

In [ ]:
def tanya_tim(tim, pertanyaan: str, konteks: KonteksPengguna) -> None:
    handler = UsageMetadataCallbackHandler()
    # TODO(B8.4): panggil tim.invoke dengan pesan pengguna, context=konteks, dan config={"callbacks": [handler]} untuk menghitung token
    response = None  # TODO: ganti dengan implementasi Anda
    ditanya = [
        panggilan["name"].replace("tanya_spesialis_", "")
        for pesan in response["messages"]
        if isinstance(pesan, AIMessage)
        for panggilan in pesan.tool_calls
    ]
    token = sum(u.get("total_tokens", 0) for u in handler.usage_metadata.values())

    print(f"[{konteks.role} @ {konteks.area}] {pertanyaan}")
    print(f"   Spesialis ditanya: {ditanya}   |   token: {token}")
    print(f"   Jawaban: {response['messages'][-1].content}\n")


engineer = KonteksPengguna(role="engineer", area="Seruni", user_id="ENG-208")
operator = KonteksPengguna(role="operator", area="Kenanga", user_id="OP-322")

## Langkah 6 - Pertanyaan 1: engineer menanyakan status sumur

In [ ]:
tanya_tim(tim, "Bagaimana status sumur SR-020?", engineer)

## Langkah 7 - Pertanyaan 2: operator menanyakan sumur bermasalah dan jadwal pulling

In [ ]:
tanya_tim(tim, "Sumur mana yang bermasalah dan kapan jadwal pulling berikutnya?", operator)

## Langkah 8 - Pertanyaan 3: engineer meminta laporan gabungan
Ketiga spesialis boleh dipakai, termasuk Engineering.

In [ ]:
tanya_tim(tim, "Beri saya status sumur, work order terbuka, dan water cut untuk 120 bopd minyak dan 480 bwpd air.", engineer)

## Langkah 9 - Pertanyaan 4: operator meminta laporan gabungan yang sama
Bagian Engineering ditolak oleh kode karena peran operator.

In [ ]:
tanya_tim(tim, "Beri saya status sumur, work order terbuka, dan water cut untuk 120 bopd minyak dan 480 bwpd air.", operator)